# FinSage — Train on a free Colab T4
**Runtime → Change runtime type → T4 GPU** before running.

Pipeline: prepare data → QLoRA SFT → build DPO pairs → DPO → merge + quantize.

Every stage auto-backs-up `outputs/`, `models/`, `data/processed/` to Google Drive,
so a Colab disconnect never loses your trained adapters.

In [ ]:
# 1. Get the project
REPO_URL = 'https://github.com/AshutoshMore96/finsage-llm-finetuning.git'
!git clone $REPO_URL finsage || echo 'clone failed — check the URL or make the repo public'
%cd finsage

In [ ]:
# 2. Install deps (unsloth pulls compatible transformers/trl/peft/bitsandbytes)
!pip install -q -r requirements.txt

In [ ]:
# 3. Mount Google Drive + define backup() — run this once.
#    /content is wiped on disconnect; Drive is persistent.
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/finsage_backup'
PROJECT_DIR = '/content/finsage'

def backup(tag=''):
    """Copy artifacts to Drive so a runtime recycle can't erase them."""
    if not os.path.exists('/content/drive/MyDrive'):
        print('⚠️  Drive not mounted — skipping backup'); return
    os.chdir(PROJECT_DIR)
    for p in ['outputs', 'models', 'data/processed']:
        if os.path.exists(p):
            dst = os.path.join(BACKUP_DIR, p)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copytree(p, dst, dirs_exist_ok=True)
    print(f'✅ backed up outputs/models/data → {BACKUP_DIR}  [{tag}]')

def restore():
    """Pull artifacts back from Drive after a fresh runtime (to resume)."""
    if not os.path.exists(BACKUP_DIR):
        print('No backup found yet'); return
    os.chdir(PROJECT_DIR)
    for p in ['outputs', 'models', 'data/processed']:
        src = os.path.join(BACKUP_DIR, p)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(p) or '.', exist_ok=True)
            shutil.copytree(src, p, dirs_exist_ok=True)
    print('✅ restored artifacts from Drive')

# If you reconnected after a disconnect, run restore() to continue where you left off:
# restore()

In [ ]:
# 4. (Optional) log in to push the final model to the Hugging Face Hub
# from huggingface_hub import notebook_login; notebook_login()
# Then set merge.push_to_hub: true and merge.hub_repo in config/config.yaml

In [ ]:
!python -m src.data.prepare_sft_data
backup('sft_data')

In [ ]:
!python -m src.train.train_sft_qlora
backup('sft')   # the 90-min step — protect it first

In [ ]:
!python -m src.data.prepare_dpo_data
backup('dpo_data')

In [ ]:
!python -m src.train.train_dpo
backup('dpo')

In [ ]:
!python -m src.merge.merge_and_quantize
backup('merged')

In [ ]:
!python -m src.eval.evaluate
backup('eval')
from IPython.display import Markdown
Markdown(open('outputs/eval_report.md').read())

## Get the model onto your Mac
Your artifacts are already in **Drive → `finsage_backup/`**. Download
`finsage_backup/models/finsage-merged-16bit-gguf/` from Drive, then on the Mac:
```bash
pip install -r requirements-mac.txt
python -m mlx_lm.convert --hf-path models/finsage-merged-16bit -q --mlx-path models/finsage-mlx-4bit
python src/serve/app_gradio.py
```

### If you got disconnected
Re-run cells 1–3, then `restore()` (cell 3 has it), then continue from whichever
stage you hadn't finished. The DPO step also resumes on its own from saved pairs.